# 🎥 Video Understanding Pipeline — Kaggle

Run the full video analysis pipeline on a **free GPU T4 x2** (2× 15 GB VRAM).
Uses **Moondream2** vision-language model to analyze video frames.

### ⚠️ IMPORTANT: Do these BEFORE running any cells
1. **Right sidebar → Accelerator → GPU T4 x2**
2. **Right sidebar → Settings → Internet → ON**
3. Then run all cells in order

> **Note**: Kaggle provides 30 hrs/week of free GPU. T4 x2 gives you two T4 GPUs (15 GB each).

In [1]:
# ── 1. Install System Libraries + Python Packages ─────────────
# libvips is required by Moondream2 model code (pyvips dependency)
!apt-get update -qq && apt-get install -y -qq libvips-dev > /dev/null 2>&1
print('✅ libvips system library installed')

# Python packages
!pip install -q fastapi uvicorn python-multipart opencv-python-headless \
    Pillow transformers einops pyngrok accelerate pyvips
print('✅ Python packages installed')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ libvips system library installed
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 1.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Python packages installed


In [2]:
# ── 2. Verify GPU ────────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print()
    print('❌ ERROR: GPU NOT ENABLED!')
    print('=' * 50)
    print('You MUST enable GPU before running:')
    print('  1. Click the ⚡ icon in the right sidebar')
    print('  2. Select GPU T4 x2')
    print('  3. Re-run all cells from the beginning')
    print('=' * 50)
    raise RuntimeError('GPU not enabled! Enable it in the sidebar and restart.')

num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({vram:.1f} GB VRAM)")
print(f"\n✅ GPU ready! Using GPU 0 for inference (~2-3 sec/frame)")

PyTorch: 2.8.0+cu126
CUDA available: False

❌ ERROR: GPU NOT ENABLED!
You MUST enable GPU before running:
  1. Click the ⚡ icon in the right sidebar
  2. Select GPU T4 x2
  3. Re-run all cells from the beginning


RuntimeError: GPU not enabled! Enable it in the sidebar and restart.

In [ ]:
# ── 3. Load Moondream2 Model ─────────────────────────────────
from transformers import AutoModelForCausalLM

DEVICE = "cuda:0"  # Use first T4 GPU
DTYPE = torch.float16

print(f"Loading Moondream2 on {DEVICE} (float16)...")
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    revision="2025-01-09",
    trust_remote_code=True,
    device_map={"": DEVICE},
    torch_dtype=DTYPE,
)
used_mb = torch.cuda.memory_allocated(0) / 1e6
print(f"✅ Model loaded on GPU 0 — using {used_mb:.0f} MB VRAM")

In [ ]:
# ── 4. Video Processing Utilities ────────────────────────────
import cv2
import numpy as np
from PIL import Image
from typing import List, Tuple

def extract_frames(video_path: str, interval_sec: float = 1.0, max_width: int = 512) -> List[Tuple[float, Image.Image]]:
    """Extract frames from video at given interval."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps <= 0:
        fps = 30.0
    frame_interval = max(int(fps * interval_sec), 1)
    duration = total / fps if total > 0 else 0
    print(f"  Video: {duration:.1f}s, {fps:.0f} fps, extracting every {interval_sec}s")
    
    results = []
    frame_idx = 0
    
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        
        timestamp = frame_idx / fps
        h, w = frame.shape[:2]
        if w > max_width:
            scale = max_width / w
            frame = cv2.resize(frame, (max_width, int(h * scale)))
        
        pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        results.append((timestamp, pil_image))
        frame_idx += frame_interval
    
    cap.release()
    print(f"  Extracted {len(results)} frames")
    return results

def analyze_video(video_path: str, prompt: str, interval_sec: float = 1.0):
    """Full pipeline: extract frames → Moondream2 → results."""
    frames = extract_frames(video_path, interval_sec)
    results = []
    for i, (ts, img) in enumerate(frames):
        mm, ss = int(ts // 60), int(ts % 60)
        timestamp = f"{mm:02d}:{ss:02d}"
        print(f"  Frame {i+1}/{len(frames)} @ {timestamp}", end=" ")
        
        answer = model.query(img, prompt)
        desc = answer.get("answer", str(answer)) if isinstance(answer, dict) else str(answer)
        print(f"→ {desc[:80]}..." if len(desc) > 80 else f"→ {desc}")
        
        results.append({
            "timestamp": timestamp,
            "seconds": round(ts, 2),
            "description": desc.strip(),
            "frame_index": i + 1,
        })
        
        torch.cuda.empty_cache()
    
    return results

print("✅ Video utilities ready!")

---

## Option A: Upload & Analyze in Notebook

Upload a video using Kaggle's `+Add Data` sidebar button.

In [ ]:
# ── 5A. Upload via Kaggle & Analyze ──────────────────────────
# Click "+Add Data" in the right sidebar to upload your video.
# It will appear under /kaggle/input/<dataset-name>/

import os, time, glob

# Auto-find video files in Kaggle's input directory
input_files = glob.glob("/kaggle/input/**/*.*", recursive=True)
video_exts = {'.mp4', '.webm', '.avi', '.mov', '.mkv'}
videos = [f for f in input_files if os.path.splitext(f)[1].lower() in video_exts]

if videos:
    print("📁 Found video files:")
    for i, v in enumerate(videos):
        size_mb = os.path.getsize(v) / (1024 * 1024)
        print(f"  [{i}] {v} ({size_mb:.1f} MB)")
    
    # ── Change these settings ──
    VIDEO_INDEX = 0  # Which video to analyze (0 = first found)
    PROMPT = "Describe what is happening in this frame in detail."
    # ──────────────────────────
    
    video_path = videos[VIDEO_INDEX]
    print(f"\n🔍 Analyzing: {os.path.basename(video_path)}")
    print(f"   Prompt: '{PROMPT}'\n")
    
    start = time.time()
    results = analyze_video(video_path, PROMPT)
    elapsed = time.time() - start
    
    print("\n" + "═" * 60)
    print(f"📊 RESULTS  ({elapsed:.1f}s on GPU T4)")
    print("═" * 60)
    for r in results:
        print(f"\n[{r['timestamp']}] {r['description']}")
else:
    print("📂 No video files found in /kaggle/input/")
    print("\n   To upload a video:")
    print("   1. Click '+Add Data' in the right sidebar")
    print("   2. Click 'Upload' tab → drag your video file")
    print("   3. Wait for upload to complete")
    print("   4. Re-run this cell")
    print("\n   Or use Option B below for a web UI with drag-and-drop.")

---

## Option B: Launch Full Web UI (ngrok tunnel)

Get a **free** ngrok token at [ngrok.com/signup](https://ngrok.com/signup) and paste below.

> **Important**: Make sure **Internet** is enabled: Settings → Internet → On

In [ ]:
# ── 5B. Configure ngrok ──────────────────────────────────────

NGROK_AUTH_TOKEN = ""  # ← Paste your free ngrok token here

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok configured")
else:
    print("⚠️  No ngrok token set!")
    print("   Get a free token at: https://ngrok.com/signup")
    print("   Paste it in NGROK_AUTH_TOKEN above and re-run.")

In [ ]:
# ── Full HTML UI (embedded) ──────────────────────────────────
# Run this cell BEFORE the server cell below.

FULL_HTML = r"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>VideoAI — Kaggle T4 GPU</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <style>
        *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
        :root {
            --bg: #0a0a0f; --card: rgba(18,18,28,.75); --card-h: rgba(25,25,40,.85);
            --border: rgba(255,255,255,.06); --border-h: rgba(255,255,255,.12);
            --text: #e8e8f0; --text2: #8888a8; --muted: #555570;
            --accent: #7c5cfc; --glow: rgba(124,92,252,.3); --green: #5cf0c8;
            --danger: #f05868; --radius: 16px; --sm: 10px;
            --transition: .25s cubic-bezier(.4,0,.2,1); --font: 'Inter',sans-serif;
        }
        body { font-family: var(--font); background: var(--bg); color: var(--text); min-height: 100vh; line-height: 1.6; }
        .glow { position: fixed; border-radius: 50%; filter: blur(120px); opacity: .4; pointer-events: none; z-index: 0; }
        .g1 { width: 500px; height: 500px; background: radial-gradient(circle, var(--accent) 0%, transparent 70%); top: -150px; right: -100px; animation: f1 20s ease-in-out infinite; }
        .g2 { width: 400px; height: 400px; background: radial-gradient(circle, var(--green) 0%, transparent 70%); bottom: -100px; left: -100px; animation: f2 25s ease-in-out infinite; }
        @keyframes f1 { 0%,100%{transform:translate(0,0)} 50%{transform:translate(-60px,40px)} }
        @keyframes f2 { 0%,100%{transform:translate(0,0)} 50%{transform:translate(40px,-60px)} }
        header { position: relative; z-index: 10; display: flex; align-items: center; justify-content: space-between; padding: 20px 32px; border-bottom: 1px solid var(--border); backdrop-filter: blur(20px); background: rgba(10,10,15,.6); }
        .logo { display: flex; align-items: center; gap: 12px; } .logo h1 { font-size: 22px; font-weight: 700; }
        .accent { color: var(--accent); }
        .badge { display: flex; align-items: center; gap: 8px; padding: 6px 14px; border-radius: 20px; background: var(--card); border: 1px solid var(--border); font-size: 13px; color: var(--text2); }
        .dot { width: 8px; height: 8px; border-radius: 50%; background: var(--muted); transition: var(--transition); }
        .dot.on { background: var(--green); box-shadow: 0 0 8px var(--green); }
        .container { position: relative; z-index: 10; display: grid; grid-template-columns: 1fr 1fr; gap: 24px; max-width: 1280px; margin: 32px auto; padding: 0 24px; }
        @media (max-width: 900px) { .container { grid-template-columns: 1fr; } }
        .panel { background: var(--card); border: 1px solid var(--border); border-radius: var(--radius); padding: 28px; backdrop-filter: blur(24px); }
        .panel:hover { border-color: var(--border-h); }
        .title { font-size: 16px; font-weight: 600; margin-bottom: 20px; }
        .upload { border: 2px dashed var(--border); border-radius: var(--sm); padding: 40px 24px; text-align: center; cursor: pointer; transition: var(--transition); margin-bottom: 20px; }
        .upload:hover { border-color: var(--accent); background: rgba(124,92,252,.05); }
        .upload-icon { font-size: 36px; margin-bottom: 8px; }
        .file-info { display: flex; align-items: center; gap: 14px; text-align: left; }
        .file-info .icon { font-size: 32px; } .file-info .name { font-size: 14px; font-weight: 500; word-break: break-all; }
        .file-info .size { font-size: 12px; color: var(--text2); }
        .file-remove { width: 32px; height: 32px; border: 1px solid var(--border); border-radius: 8px; background: none; color: var(--text2); cursor: pointer; display: flex; align-items: center; justify-content: center; }
        .file-remove:hover { background: rgba(240,88,104,.1); border-color: var(--danger); color: var(--danger); }
        .prompt { width: 100%; padding: 12px 16px; border: 1px solid var(--border); border-radius: var(--sm); background: rgba(255,255,255,.03); color: var(--text); font-family: var(--font); font-size: 14px; resize: vertical; transition: var(--transition); margin-bottom: 20px; }
        .prompt:focus { outline: none; border-color: var(--accent); box-shadow: 0 0 0 3px var(--glow); }
        .prompt::placeholder { color: var(--muted); }
        label { display: block; font-size: 13px; font-weight: 500; color: var(--text2); margin-bottom: 8px; }
        .btn { width: 100%; padding: 14px; border: none; border-radius: var(--sm); background: linear-gradient(135deg, var(--accent), #9c7cfc); color: #fff; font-size: 15px; font-weight: 600; cursor: pointer; transition: var(--transition); font-family: var(--font); }
        .btn:hover:not(:disabled) { transform: translateY(-2px); box-shadow: 0 8px 30px var(--glow); }
        .btn:disabled { opacity: .4; cursor: not-allowed; }
        .error { margin-top: 12px; padding: 10px 16px; border-radius: var(--sm); background: rgba(240,88,104,.1); border: 1px solid rgba(240,88,104,.2); color: var(--danger); font-size: 13px; display: none; }
        .empty { text-align: center; padding: 60px 20px; color: var(--muted); } .empty .icon { font-size: 48px; opacity: .5; margin-bottom: 16px; }
        .loading { text-align: center; padding: 60px 20px; display: none; }
        .dots { display: flex; justify-content: center; gap: 6px; margin-bottom: 20px; }
        .dots span { width: 14px; height: 14px; border-radius: 50%; background: var(--accent); animation: bounce 1.4s ease-in-out infinite; }
        .dots span:nth-child(2) { animation-delay: .16s; } .dots span:nth-child(3) { animation-delay: .32s; }
        @keyframes bounce { 0%,80%,100%{transform:scale(.6);opacity:.4} 40%{transform:scale(1);opacity:1} }
        .meta { display: grid; grid-template-columns: repeat(3,1fr); gap: 12px; margin-bottom: 14px; }
        .meta-item { background: rgba(255,255,255,.03); border: 1px solid var(--border); border-radius: 10px; padding: 12px; text-align: center; }
        .meta-label { display: block; font-size: 11px; font-weight: 500; color: var(--muted); text-transform: uppercase; margin-bottom: 4px; }
        .meta-val { font-size: 18px; font-weight: 700; color: var(--green); }
        .results-list { display: flex; flex-direction: column; gap: 10px; max-height: 520px; overflow-y: auto; }
        .result-card { display: flex; gap: 14px; padding: 14px 16px; border-radius: var(--sm); background: rgba(255,255,255,.02); border: 1px solid var(--border); animation: fadeUp .4s ease-out both; }
        .result-card:hover { background: var(--card-h); border-color: var(--border-h); }
        @keyframes fadeUp { from{opacity:0;transform:translateY(12px)} to{opacity:1;transform:translateY(0)} }
        .ts { flex-shrink: 0; padding: 4px 10px; border-radius: 6px; background: var(--glow); color: var(--accent); font-size: 12px; font-weight: 600; height: fit-content; }
        .desc { font-size: 14px; line-height: 1.6; }
        .history { position: relative; z-index: 10; max-width: 1280px; margin: 32px auto 0; padding: 0 24px; }
        .hist-header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; }
        .hist-list { display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr)); gap: 12px; }
        .hist-card { padding: 16px; background: var(--card); border: 1px solid var(--border); border-radius: var(--sm); cursor: pointer; transition: var(--transition); }
        .hist-card:hover { border-color: var(--accent); transform: translateY(-2px); box-shadow: 0 8px 24px rgba(124,92,252,.1); }
        .hist-top { display: flex; justify-content: space-between; margin-bottom: 8px; }
        .hist-name { font-size: 14px; font-weight: 600; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; max-width: 85%; }
        .hist-del { background: none; border: none; cursor: pointer; opacity: .4; font-size: 14px; } .hist-del:hover { opacity: 1; }
        .hist-prompt { font-size: 13px; color: var(--text2); overflow: hidden; text-overflow: ellipsis; white-space: nowrap; margin-bottom: 10px; }
        .hist-meta { display: flex; flex-wrap: wrap; gap: 8px; font-size: 11px; color: var(--muted); }
        .hist-meta span { padding: 2px 8px; border-radius: 4px; background: rgba(255,255,255,.03); border: 1px solid var(--border); }
        .refresh { width: 36px; height: 36px; border: 1px solid var(--border); border-radius: 8px; background: var(--card); color: var(--text2); cursor: pointer; transition: var(--transition); display: flex; align-items: center; justify-content: center; }
        .refresh:hover { color: var(--accent); transform: rotate(90deg); }
        .kaggle-badge { background: linear-gradient(135deg, #76B900, #4CAF50); color: white; padding: 4px 12px; border-radius: 12px; font-size: 11px; font-weight: 600; }
        footer { position: relative; z-index: 10; text-align: center; padding: 24px; color: var(--muted); font-size: 13px; border-top: 1px solid var(--border); margin-top: 40px; }
        footer strong { color: var(--accent); }
        .hidden { display: none !important; }
    </style>
</head>
<body>
    <div class="glow g1"></div><div class="glow g2"></div>
    <header>
        <div class="logo"><span style="font-size:28px">🎥</span><h1>Video<span class="accent">AI</span></h1></div>
        <div style="display:flex;gap:12px;align-items:center">
            <span class="kaggle-badge">KAGGLE T4 x2</span>
            <div class="badge"><span class="dot" id="dot"></span><span id="status">Connecting...</span></div>
        </div>
    </header>
    <main class="container">
        <section class="panel">
            <h2 class="title">📤 Input</h2>
            <div class="upload" id="dropZone" onclick="document.getElementById('fileInput').click()">
                <div id="uploadContent"><div class="upload-icon">☁️</div><p style="font-size:15px;font-weight:500">Drop video or click to upload</p><p style="font-size:13px;color:var(--text2)">MP4, WebM, AVI, MOV</p><p style="font-size:12px;color:var(--muted)">Max 20 MB</p></div>
                <div id="fileSelected" class="file-info hidden"></div>
            </div>
            <input type="file" id="fileInput" accept=".mp4,.webm,.avi,.mov,.mkv" hidden>
            <div><label>Prompt</label><textarea class="prompt" id="prompt" rows="3" placeholder="Describe what is happening in this video..."></textarea></div>
            <button class="btn" id="analyzeBtn" disabled>⚡ Analyze Video</button>
            <div class="error" id="error"></div>
        </section>
        <section class="panel">
            <h2 class="title">📊 Results</h2>
            <div id="empty" class="empty"><div class="icon">🔍</div><p>Upload a video and enter a prompt to begin</p></div>
            <div class="loading" id="loading"><div class="dots"><span></span><span></span><span></span></div><p style="font-size:15px;font-weight:500">Analyzing frames...</p><p style="font-size:13px;color:var(--muted)">T4 GPU • ~2-3s per frame</p></div>
            <div id="resultsMeta"></div>
            <div class="results-list" id="resultsList"></div>
        </section>
    </main>
    <section class="history" id="historySection">
        <div class="hist-header"><h2 class="title">📜 Analysis History</h2><button class="refresh" onclick="loadHistory()">🔄</button></div>
        <div class="hist-list" id="histList"><div class="empty"><div class="icon">📂</div><p>No saved analyses yet</p></div></div>
    </section>
    <footer><p>Powered by <strong>Moondream2</strong> • Running on Kaggle T4 x2 GPU</p></footer>
<script>
const fileInput=document.getElementById('fileInput'),prompt=document.getElementById('prompt'),analyzeBtn=document.getElementById('analyzeBtn');
const dropZone=document.getElementById('dropZone'),uploadContent=document.getElementById('uploadContent'),fileSelected=document.getElementById('fileSelected');
const errorEl=document.getElementById('error'),emptyEl=document.getElementById('empty'),loadingEl=document.getElementById('loading');
const resultsMeta=document.getElementById('resultsMeta'),resultsList=document.getElementById('resultsList');
const dot=document.getElementById('dot'),statusEl=document.getElementById('status'),histList=document.getElementById('histList');
let selectedFile=null;

fetch('/api/health').then(r=>r.json()).then(d=>{dot.classList.add('on');statusEl.textContent=d.gpu_name||d.device;}).catch(()=>{statusEl.textContent='Offline';});

function setFile(f){if(!f)return;selectedFile=f;uploadContent.classList.add('hidden');fileSelected.classList.remove('hidden');fileSelected.innerHTML=`<span class="icon">🎬</span><div style="flex:1"><div class="name">${esc(f.name)}</div><div class="size">${(f.size/1e6).toFixed(1)} MB</div></div><button class="file-remove" onclick="event.stopPropagation();clearFile()">✕</button>`;updateBtn();}
function clearFile(){selectedFile=null;fileInput.value='';uploadContent.classList.remove('hidden');fileSelected.classList.add('hidden');updateBtn();}
function updateBtn(){analyzeBtn.disabled=!(selectedFile&&prompt.value.trim());}
function esc(s){const d=document.createElement('div');d.textContent=s;return d.innerHTML;}

fileInput.addEventListener('change',e=>{if(e.target.files[0])setFile(e.target.files[0]);});
prompt.addEventListener('input',updateBtn);
dropZone.addEventListener('dragover',e=>{e.preventDefault();dropZone.style.borderColor='var(--accent)';});
dropZone.addEventListener('dragleave',()=>{dropZone.style.borderColor='';});
dropZone.addEventListener('drop',e=>{e.preventDefault();dropZone.style.borderColor='';if(e.dataTransfer.files[0])setFile(e.dataTransfer.files[0]);});

analyzeBtn.addEventListener('click',async()=>{
    errorEl.style.display='none';emptyEl.classList.add('hidden');loadingEl.style.display='block';
    resultsMeta.innerHTML='';resultsList.innerHTML='';analyzeBtn.disabled=true;
    const fd=new FormData();fd.append('video',selectedFile);fd.append('prompt',prompt.value);
    try{
        const r=await fetch('/api/analyze',{method:'POST',body:fd});
        const d=await r.json();if(!r.ok)throw new Error(d.detail||'Analysis failed');
        displayResults(d);
    }catch(e){errorEl.textContent=e.message;errorEl.style.display='block';emptyEl.classList.remove('hidden');}
    finally{loadingEl.style.display='none';analyzeBtn.disabled=false;updateBtn();loadHistory();}
});

function displayResults(d){
    emptyEl.classList.add('hidden');loadingEl.style.display='none';
    resultsMeta.innerHTML=`<div class="meta"><div class="meta-item"><span class="meta-label">Duration</span><span class="meta-val">${d.video_duration_seconds||0}s</span></div><div class="meta-item"><span class="meta-label">Frames</span><span class="meta-val">${d.frames_analyzed}</span></div><div class="meta-item"><span class="meta-label">Time</span><span class="meta-val">${d.processing_time_seconds}s</span></div></div>`;
    resultsList.innerHTML=d.results.map(r=>`<div class="result-card"><span class="ts">${r.timestamp}</span><span class="desc">${esc(r.description)}</span></div>`).join('');
}

async function loadHistory(){try{const r=await fetch('/api/history');const d=await r.json();const a=d.analyses||[];if(!a.length){histList.innerHTML='<div class="empty"><div class="icon">📂</div><p>No saved analyses yet</p></div>';return;}histList.innerHTML=a.map(h=>`<div class="hist-card" onclick="viewHist('${h.id}')"><div class="hist-top"><span class="hist-name">🎬 ${esc(h.video_filename||'')}</span><button class="hist-del" onclick="event.stopPropagation();delHist('${h.id}')">🗑️</button></div><div class="hist-prompt">${esc(h.prompt||'')}</div><div class="hist-meta"><span>${h.frames_analyzed} frames</span><span>${h.processing_time_seconds}s</span><span>${(h.device||'').toUpperCase()}</span></div></div>`).join('');}catch(e){console.error(e);}}
async function viewHist(id){try{const r=await fetch(`/api/history/${id}`);const d=await r.json();displayResults(d);resultsMeta.scrollIntoView({behavior:'smooth'});}catch(e){errorEl.textContent='Failed to load';errorEl.style.display='block';}}
async function delHist(id){if(!confirm('Delete?'))return;try{await fetch(`/api/history/${id}`,{method:'DELETE'});loadHistory();}catch(e){}}
loadHistory();
</script>
</body></html>
"""
print("✅ UI template loaded. Run the server cell below.")

In [ ]:
# ── 5B. Launch FastAPI Server + ngrok ─────────────────────────
import os, uuid, time, json, threading
from datetime import datetime
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Video Understanding Pipeline — Kaggle T4 x2")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

MAX_SIZE = 20 * 1024 * 1024
HISTORY = []  # In-memory history for session

@app.get("/api/health")
async def health():
    gpus = []
    for i in range(torch.cuda.device_count()):
        gpus.append({"id": i, "name": torch.cuda.get_device_name(i)})
    return {
        "status": "ok", "device": DEVICE, "model_loaded": True,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "gpus": gpus,
    }

@app.post("/api/analyze")
async def api_analyze(video: UploadFile = File(...), prompt: str = Form(...)):
    start = time.time()
    content = await video.read()
    if len(content) > MAX_SIZE:
        raise HTTPException(413, "File too large. Max 20 MB.")
    
    ext = os.path.splitext(video.filename or '.mp4')[1]
    path = f"/tmp/{uuid.uuid4().hex}{ext}"
    try:
        with open(path, "wb") as f:
            f.write(content)
        
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total / fps if total > 0 and fps > 0 else 0
        cap.release()
        
        results = analyze_video(path, prompt)
        elapsed = time.time() - start
        
        record = {
            "success": True,
            "id": uuid.uuid4().hex[:12],
            "prompt": prompt,
            "video_filename": video.filename or "video",
            "video_duration_seconds": round(duration, 2),
            "frames_analyzed": len(results),
            "processing_time_seconds": round(elapsed, 2),
            "device": DEVICE,
            "created_at": datetime.now().isoformat(),
            "results": results,
        }
        HISTORY.insert(0, record)
        return record
    finally:
        if os.path.exists(path):
            os.remove(path)

@app.get("/api/history")
async def get_history():
    summaries = [{k: v for k, v in h.items() if k != 'results'} for h in HISTORY]
    return {"analyses": summaries}

@app.get("/api/history/{analysis_id}")
async def get_history_item(analysis_id: str):
    for h in HISTORY:
        if h["id"] == analysis_id:
            return h
    raise HTTPException(404, "Not found")

@app.delete("/api/history/{analysis_id}")
async def delete_history_item(analysis_id: str):
    for i, h in enumerate(HISTORY):
        if h["id"] == analysis_id:
            HISTORY.pop(i)
            return {"success": True}
    raise HTTPException(404, "Not found")

@app.get("/", response_class=HTMLResponse)
async def serve_ui():
    return FULL_HTML

# Start server
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(3)

if NGROK_AUTH_TOKEN:
    public_url = ngrok.connect(8000)
    print(f"\n🌐 Web UI: {public_url}")
    print(f"   Health:  {public_url}/api/health")
    print(f"\n💡 Open the URL above in your browser!")
    print(f"   Kaggle T4 x2 GPU gives you ~2-3 sec/frame inference.")
else:
    print("\n⚠️  No ngrok token. Set NGROK_AUTH_TOKEN above and re-run.")

---

## 📊 Performance Comparison

| Platform | GPU | VRAM | Per-Frame | 7 Frames |
|----------|-----|------|-----------|----------|
| Local (CPU) | — | — | ~4 min | ~28 min |
| Local (MX450) | MX450 | 2 GB | ~1 min | ~7 min |
| **Kaggle T4 x2** | **T4** | **2× 15 GB** | **~2-3 sec** | **~15 sec** |
| Colab | T4 | 15 GB | ~2-3 sec | ~15 sec |